In [9]:
!pip install ipywidgets --quiet

import pandas as pd
from IPython.display import display, HTML, clear_output
import ipywidgets as widgets


In [10]:
# Sample recipes CSV generator (run once, or modify as needed)

data = [
    # meal_type, name, goal_tags, diet_tags, calories, ingredients, steps
    ["breakfast", "Vegan Smoothie Bowl",
     "Lose Weight,Gan Muscle", "Vegan,Gluten-free", 320,
     "frozen berries;banana;soy yogurt;granola;chia seeds",
     "Blend berries and banana with soy yogurt; top with granola and chia seeds."],

    ["breakfast", "Greek Yogurt Parfait",
     "Lose Weight", "Vegetarian", 300,
     "greek yogurt;berries;honey;granola",
     "Layer yogurt, berries, honey, and granola in a glass."],

    ["breakfast", "Egg Omelette with Toast",
     "Gain Muscle", "None", 450,
     "eggs;cheese;spinach;whole grain bread",
     "Whisk eggs, cook with spinach and cheese, serve with toast."],

    ["lunch", "Chickpea & Quinoa Salad",
     "Lose Weight", "Vegan,Gluten-free", 480,
     "quinoa;chickpeas;cucumber;tomato;olive oil;lemon",
     "Mix cooked quinoa & chickpeas, add chopped veggies, dress with olive oil & lemon."],

    ["lunch", "Grilled Chicken Salad",
     "Lose Weight,Gan Muscle", "None,Gluten-free", 520,
     "chicken breast;lettuce;tomato;cucumber;olive oil",
     "Grill chicken, slice and serve on salad with simple dressing."],

    ["lunch", "Tofu Power Bowl",
     "Gain Muscle", "Vegan", 550,
     "tofu;brown rice;broccoli;soy sauce;sesame seeds",
     "Stir-fry tofu and broccoli, serve over rice, drizzle with soy sauce."],

    ["dinner", "Lentil Veggie Stew",
     "Lose Weight,Gan Muscle", "Vegan,Gluten-free", 500,
     "lentils;carrots;celery;tomatoes;spices",
     "Simmer lentils with chopped vegetables and spices until tender."],

    ["dinner", "Baked Salmon & Veggies",
     "Lose Weight,Gan Muscle", "None,Gluten-free", 600,
     "salmon;asparagus;olive oil;lemon;garlic",
     "Bake salmon and asparagus with olive oil, lemon, and garlic."],

    ["dinner", "Paneer & Veggie Stir-fry",
     "Gain Muscle", "Vegetarian", 650,
     "paneer;peppers;onions;soy sauce;rice",
     "Stir-fry paneer and veggies, serve with rice."],
]

df_sample = pd.DataFrame(data, columns=[
    "meal_type", "name", "goal_tags", "diet_tags",
    "calories", "ingredients", "steps"
])

csv_path = "/content/recipes.csv"
df_sample.to_csv(csv_path, index=False)
print(f"Sample recipes.csv written to: {csv_path}")
df_sample.head()


Sample recipes.csv written to: /content/recipes.csv


,meal_type,name,goal_tags,diet_tags,calories,ingredients,steps
0,breakfast,Vegan Smoothie Bowl,"Lose Weight,Gan Muscle","Vegan,Gluten-free",320,frozen berries;banana;soy yogurt;granola;chia ...,Blend berries and banana with soy yogurt; top ...
1,breakfast,Greek Yogurt Parfait,Lose Weight,Vegetarian,300,greek yogurt;berries;honey;granola,"Layer yogurt, berries, honey, and granola in a..."
2,breakfast,Egg Omelette with Toast,Gain Muscle,None,450,eggs;cheese;spinach;whole grain bread,"Whisk eggs, cook with spinach and cheese, serv..."
3,lunch,Chickpea & Quinoa Salad,Lose Weight,"Vegan,Gluten-free",480,quinoa;chickpeas;cucumber;tomato;olive oil;lemon,"Mix cooked quinoa & chickpeas, add chopped veg..."
4,lunch,Grilled Chicken Salad,"Lose Weight,Gan Muscle","None,Gluten-free",520,chicken breast;lettuce;tomato;cucumber;olive oil,"Grill chicken, slice and serve on salad with s..."


In [11]:
# Load recipes from CSV
csv_path = "/content/recipes.csv"  # change if your file is elsewhere
recipes_df = pd.read_csv(csv_path)

print("Loaded recipes:")
display(recipes_df.head())


Loaded recipes:


,meal_type,name,goal_tags,diet_tags,calories,ingredients,steps
0,breakfast,Vegan Smoothie Bowl,"Lose Weight,Gan Muscle","Vegan,Gluten-free",320,frozen berries;banana;soy yogurt;granola;chia ...,Blend berries and banana with soy yogurt; top ...
1,breakfast,Greek Yogurt Parfait,Lose Weight,Vegetarian,300,greek yogurt;berries;honey;granola,"Layer yogurt, berries, honey, and granola in a..."
2,breakfast,Egg Omelette with Toast,Gain Muscle,NaN,450,eggs;cheese;spinach;whole grain bread,"Whisk eggs, cook with spinach and cheese, serv..."
3,lunch,Chickpea & Quinoa Salad,Lose Weight,"Vegan,Gluten-free",480,quinoa;chickpeas;cucumber;tomato;olive oil;lemon,"Mix cooked quinoa & chickpeas, add chopped veg..."
4,lunch,Grilled Chicken Salad,"Lose Weight,Gan Muscle","None,Gluten-free",520,chicken breast;lettuce;tomato;cucumber;olive oil,"Grill chicken, slice and serve on salad with s..."


In [12]:
from typing import Optional, Dict, Any, List

def get_recipe_for_meal(
    df: pd.DataFrame, goal: str, restriction: str, meal_type: str
) -> Optional[Dict[str, Any]]:
    """
    Returns a dict with recipe info for the requested meal_type,
    filtered by goal + restriction (using simple 'contains' checks
    on goal_tags and diet_tags).
    If nothing matches, returns None.
    """
    # Ensure clean strings
    goal = goal.strip()
    restriction = restriction.strip()

    subset = df[df["meal_type"].str.lower() == meal_type.lower()].copy()

    # Filter on goal: goal_tags contains selected goal
    subset = subset[subset["goal_tags"].fillna("").str.contains(goal, case=False)]

    # Filter on restriction, unless it's "None"
    if restriction.lower() != "none":
        subset = subset[subset["diet_tags"].fillna("").str.contains(restriction, case=False)]

    if subset.empty:
        return None

    # Just pick the first match (simple MVP)
    row = subset.iloc[0]

    return {
        "name": row["name"],
        "calories": row.get("calories", None),
        "ingredients": row.get("ingredients", ""),
        "steps": row.get("steps", ""),
        "diet_tags": row.get("diet_tags", ""),
        "goal_tags": row.get("goal_tags", ""),
    }


def get_grocery_stores(zip_code: str) -> List[str]:
    """
    Simple offline mock of grocery store locations.
    - If zip starts with '1' → city-like stores
    - If zip starts with '9' → west coast-like stores
    - Else → generic local stores
    """
    zip_code = str(zip_code).strip()
    if not zip_code:
        return ["(Enter a valid ZIP to see grocery stores.)"]

 #   first_char = zip_code[0]

    if zip_code == "08610":
        return [
            "Walmart — 0.7 mi, $$",
            "Dollar General — 1.1 mi, $",
            "ShopRite Pharmacy of Halmilton Marketplace — 0.5 mi, $$",
        ]
    elif zip_code == "08810":
        return [
            "Dorothy Lane Market — 0.6 mi, $$$",
            "City Market — 1.2 mi, $$",
            "Grocery Outlet — 0.6 mi, $",
            "Dot's Supermarket — 1.2 mi, $",
            "Trader Joe's — 0.6 mi, $$$",
        ]
    else:
        return [
            "New Dodges Market — 0.8 mi, $$",
            "Elmer IGA — 1.3 mi, $",
            "Super Value of Elmer — 0.6 mi, $$$",
            "DG Market — 1.2 mi, $",
        ]


In [13]:
# --- Widgets ---

zip_widget = widgets.Text(
    value="10001",
    placeholder="Enter ZIP code",
    description="ZIP:",
    disabled=False,
)

goal_widget = widgets.Dropdown(
    options=["Lose Weight", "Gain Muscle"],
    value="Lose Weight",
    description="Goal:",
)

restriction_widget = widgets.Dropdown(
    options=["None", "Vegan", "Vegetarian", "Gluten-free"],
    value="None",
    description="Diet:",
)

generate_button = widgets.Button(
    description="Generate Plan",
    button_style="success",
    tooltip="Click to generate meal plan",
    icon="check"
)

output_area = widgets.Output()


In [14]:
def on_generate_clicked(b):
    with output_area:
        clear_output()

        zip_code = zip_widget.value.strip()
        goal = goal_widget.value
        restriction = restriction_widget.value

        if not zip_code:
            display(HTML("<b style='color:red;'>Please enter a ZIP code.</b>"))
            return

        # Get recipes
        breakfast = get_recipe_for_meal(recipes_df, goal, restriction, "breakfast")
        lunch = get_recipe_for_meal(recipes_df, goal, restriction, "lunch")
        dinner = get_recipe_for_meal(recipes_df, goal, restriction, "dinner")

        stores = get_grocery_stores(zip_code)

        # Build HTML output
        html_parts = []

        header = f"""
        <h2>🥗 SmartMeal Plan</h2>
        <p><b>ZIP:</b> {zip_code} &nbsp;&nbsp;
           <b>Goal:</b> {goal} &nbsp;&nbsp;
           <b>Diet:</b> {restriction}
        </p>
        <hr>
        """
        html_parts.append(header)

        def format_recipe_block(title: str, recipe: Optional[Dict[str, Any]]) -> str:
            if recipe is None:
                return f"<h3>{title}</h3><p>No matching recipe found.</p>"
            ing_list = "<br>".join(
                [f"- {i.strip()}" for i in str(recipe.get('ingredients', '')).split(";") if i.strip()]
            )
            steps = recipe.get("steps", "")
            cal = recipe.get("calories", "")
            return f"""
                <h3>{title}: {recipe['name']}</h3>
                <p><b>Calories:</b> {cal}</p>
                <p><b>Ingredients:</b><br>{ing_list}</p>
                <p><b>Steps:</b> {steps}</p>
                <hr>
            """

        html_parts.append(format_recipe_block("Breakfast", breakfast))
        html_parts.append(format_recipe_block("Lunch", lunch))
        html_parts.append(format_recipe_block("Dinner", dinner))

        # Grocery stores section
        stores_html = "<h3>Nearby Grocery Stores</h3><ul>"
        for s in stores:
            stores_html += f"<li>{s}</li>"
        stores_html += "</ul>"

        html_parts.append(stores_html)

        full_html = "<div>" + "\n".join(html_parts) + "</div>"
        display(HTML(full_html))


generate_button.on_click(on_generate_clicked)


In [16]:
ui = widgets.VBox([
    widgets.HTML("<h2>🥗 Smart Meal Planner 🛒 </h2>"),
    zip_widget,
    goal_widget,
    restriction_widget,
    generate_button,
    widgets.HTML("<hr>"),
    output_area
])

display(ui)
